# Exercise 5.3 — Decision trees, overfitting and tuning (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day5/day5-03-decision-trees.html>.*

*Open it in Jupyter, VS Code or Positron and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

A decision tree classifies a person by asking a sequence of yes-or-no questions:
is glucose above 128? If so, is body mass above 30? Each answer sends the person
down one branch, until they reach a **leaf**, which gives a predicted class and
a probability. Trees are easy to read, and they can model interactions and
non-linear effects without being told to. Their weakness is that a tree can keep
splitting until it has memorised the training rows. In this exercise we fit a
tree, watch it overfit, restrain it by hand, and then let a grid search choose
the restraint for us.

**Theory:** splits, nodes and leaves; tree depth and minimum node size as
hyperparameters; overfitting in trees; grid search with cross-validation; the
test set is used once, at the end.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix, roc_auc_score

The same 750 people and split as Exercise 5.2, from
[`data/diabetes_clean.csv`](data/diabetes_clean.csv), with the five predictors pregnant,
glucose, mass, pedigree and age (triceps, insulin and blood pressure are left
out because a zero there means "not measured"). The outcome is 1 for `pos` and
0 for `neg`.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["pregnant", "glucose", "mass", "pedigree", "age", "diabetes"]]
diabetes["diabetes"] = (diabetes["diabetes"] == "pos").astype(int)       # 1 = pos, 0 = neg

is_test = diabetes.index % 5 == 0               # every fifth person is held back
train = diabetes[~is_test]
test  = diabetes[is_test]

predictors = ["pregnant", "glucose", "mass", "pedigree", "age"]

------------------------------------------------------------------------

### Part 1 — A basic decision tree

A `DecisionTreeClassifier()` with its default settings, fitted on the five
predictors. `random_state` fixes how ties between equally good splits are
broken, so that the tree is the same every time.

In [ ]:
tree_basic = DecisionTreeClassifier(random_state=123)
tree_basic = tree_basic.fit(train[predictors], train["diabetes"])

print("depth: ", tree_basic.get_depth())
print("leaves:", tree_basic.get_n_leaves())

The default tree keeps splitting until every leaf is pure: it has grown to a
depth of 18 with 151 leaves for 600 people. Draw the top three levels
(`max_depth=3` only limits the drawing, not the tree):

In [ ]:
fig, ax = plt.subplots(figsize=(22, 8))
drawn = plot_tree(tree_basic, max_depth=3, feature_names=predictors,
                  class_names=["neg", "pos"], filled=True, fontsize=8, ax=ax)
plt.show()

In each box: the split, the Gini impurity (0 for a pure node), the number of
people (`samples`), how many of them are without and with diabetes (`value`),
and the predicted class. The first split is on glucose: everything else depends
on it.

scikit-learn's defaults grow the tree until every leaf is pure. R's rpart stops
early by default (a split is only kept if it improves the fit by at least 1%),
so in the R tab the same data give a much smaller tree.

> **Learning points**
>
> - A tree splits the data with one yes-or-no question at a time; each leaf gives a predicted class and a probability.
> - The first split is on the most useful predictor, here glucose.
> - Default settings differ between packages: rpart stops early, scikit-learn grows the tree until the leaves are pure.

------------------------------------------------------------------------

### Part 2 — Does a tree that nothing stops overfit?

Predict the training and the test rows. `predict()` uses the threshold 0.5 for
us:

In [ ]:
train_preds_basic = train.copy()
train_preds_basic["prob"] = tree_basic.predict_proba(train[predictors])[:, 1]
train_preds_basic["pred_class"] = tree_basic.predict(train[predictors])

test_preds_basic = test.copy()
test_preds_basic["prob"] = tree_basic.predict_proba(test[predictors])[:, 1]
test_preds_basic["pred_class"] = tree_basic.predict(test[predictors])

confusion_matrix(train_preds_basic["diabetes"], train_preds_basic["pred_class"])

The same metrics as Exercise 5.2. Remember that `confusion_matrix()` puts the
truth in the rows. On the training rows the tree makes no mistakes at all:

In [ ]:
truth, predicted = train_preds_basic["diabetes"], train_preds_basic["pred_class"]

print("accuracy:   ", round(accuracy_score(truth, predicted), 3))
print("precision:  ", round(precision_score(truth, predicted), 3))
print("sensitivity:", round(recall_score(truth, predicted), 3))
print("specificity:", round(recall_score(truth, predicted, pos_label=0), 3))

Now the test rows:

> **Worked answer**

In [ ]:
truth, predicted = test_preds_basic["diabetes"], test_preds_basic["pred_class"]

print(confusion_matrix(truth, predicted))
print("accuracy:   ", round(accuracy_score(truth, predicted), 3))
print("precision:  ", round(precision_score(truth, predicted), 3))
print("sensitivity:", round(recall_score(truth, predicted), 3))
print("specificity:", round(recall_score(truth, predicted, pos_label=0), 3))

> Training accuracy 1.000 in both languages; test accuracy 0.673 in R and 0.693 in
> Python (the two trees differ in how ties between equally good splits are
> broken). The test sensitivity is 0.434 in R and 0.509 in Python: the tree
> misses about half of the people with diabetes it has never seen.

**Question!** The tree is perfect on the training rows and much worse on the
test rows. What has it learned that does not carry over to new people?

> **Learning points**
>
> - A tree that is allowed to keep splitting fits the training rows almost perfectly and does worse on the test rows: it overfits.
> - The bigger the gap between training and test performance, the more the model has memorised.

------------------------------------------------------------------------

### Part 3 — Restraining the tree by hand

A tree has **hyperparameters**: settings chosen before fitting that control how
complex it can become. In scikit-learn:

- `max_depth`: the maximum number of questions from the root to a leaf;
- `min_samples_split`: the minimum number of people a node must hold before it
  may be split;
- `ccp_alpha`: how much a split must improve the fit to be kept.

Set the first two by hand, to a shallow tree:

In [ ]:
tree_shallow = DecisionTreeClassifier(max_depth=3, min_samples_split=20, random_state=123)
tree_shallow = tree_shallow.fit(train[predictors], train["diabetes"])

fig, ax = plt.subplots(figsize=(14, 7))
drawn = plot_tree(tree_shallow, feature_names=predictors,
                  class_names=["neg", "pos"], filled=True, fontsize=9, ax=ax)
plt.show()

The shallow tree is small enough to read as a clinical rule. Most of its leaves
predict `neg`; only people with high glucose and a high body mass are predicted
to have diabetes. Evaluate it on both sets:

In [ ]:
for name, data in [("train", train), ("test", test)]:
    truth = data["diabetes"]
    predicted = tree_shallow.predict(data[predictors])
    print(name,
          "accuracy", round(accuracy_score(truth, predicted), 3),
          "precision", round(precision_score(truth, predicted), 3),
          "sensitivity", round(recall_score(truth, predicted), 3),
          "specificity", round(recall_score(truth, predicted, pos_label=0), 3))

Training accuracy 0.757 in R and 0.760 in Python, test accuracy 0.733 in both
(rpart also refuses to make a leaf with fewer than a third of `min_n` people,
scikit-learn does not, so the two trees differ slightly). Compared with the tree
that nothing stopped, training accuracy has dropped, because the model is
simpler, and test accuracy has risen: the shallow tree generalises better. Try another setting yourself: a tree of depth
5 that may only split nodes holding at least 40 people.

> **Worked answer**

In [ ]:
tree_try = DecisionTreeClassifier(max_depth=5, min_samples_split=40, random_state=123)
tree_try = tree_try.fit(train[predictors], train["diabetes"])

print("train accuracy:", round(accuracy_score(train["diabetes"], tree_try.predict(train[predictors])), 3))
print("test accuracy: ", round(accuracy_score(test["diabetes"], tree_try.predict(test[predictors])), 3))

> R: training accuracy 0.783, test accuracy 0.780. Python: training accuracy
> 0.785, test accuracy 0.767. Both do better on the test rows than the depth-3
> tree.

> **Learning points**
>
> - Hyperparameters such as the maximum depth and the minimum node size control how complex a tree can become.
> - A simpler tree fits the training rows less well but can generalise better.
> - Trying settings one by one, and judging them on the test rows, does not scale and spends the test set.

------------------------------------------------------------------------

### Part 4 — Tuning the hyperparameters with a grid search

Instead of guessing, try a **grid** of values and judge each combination
without touching the test rows. We split the **training** rows into 5 parts
(**folds**); each combination is fitted 5 times, each time on four folds and
evaluated on the fifth, and the 5 results are averaged. This is **5-fold
cross-validation**, and `GridSearchCV()` does all of it.

In [ ]:
tree_grid = {"max_depth":         [1, 3, 5, 7, 10],
             "min_samples_split": [2, 9, 16, 23, 30]}

Five depths times five node sizes: 25 combinations, each fitted on 5 folds.
`cv=5` makes the folds, keeping the share of people with diabetes the same in
every fold; `scoring="roc_auc"` judges each combination by its AUC.

In [ ]:
tree_search = GridSearchCV(DecisionTreeClassifier(random_state=123),
                           param_grid=tree_grid, cv=5, scoring="roc_auc")
tree_search = tree_search.fit(train[predictors], train["diabetes"])

tree_results = pd.DataFrame(tree_search.cv_results_)
tree_results = tree_results[["param_max_depth", "param_min_samples_split",
                             "mean_test_score", "std_test_score"]]
tree_results.sort_values("mean_test_score", ascending=False).head(5).round(3)

In [ ]:
fig, ax = plt.subplots()
for split in tree_grid["min_samples_split"]:
    rows = tree_results[tree_results["param_min_samples_split"] == split]
    ax.plot(rows["param_max_depth"], rows["mean_test_score"], marker="o", label=str(split))
ax.set_xlabel("max_depth")
ax.set_ylabel("Mean cross-validated AUC")
ax.legend(title="min_samples_split")
plt.show()

A depth of 1 (a single split on glucose) is clearly too simple, and so are deep
trees that may split small nodes: the AUC falls as the tree gets deeper, unless
`min_samples_split` is large. Select the best combination; `GridSearchCV()` has
already refitted it on all 600 training rows (`refit=True` is the default):

In [ ]:
print(tree_search.best_params_)
print("cross-validated AUC:", round(tree_search.best_score_, 3))

final_tree = tree_search.best_estimator_
print("leaves:", final_tree.get_n_leaves())

Now, and only now, use the test rows: evaluate the chosen tree once.

> **Worked answer**

In [ ]:
test_prob  = final_tree.predict_proba(test[predictors])[:, 1]
test_class = final_tree.predict(test[predictors])
train_prob = final_tree.predict_proba(train[predictors])[:, 1]

print("test accuracy:   ", round(accuracy_score(test["diabetes"], test_class), 3))
print("test sensitivity:", round(recall_score(test["diabetes"], test_class), 3))
print("test specificity:", round(recall_score(test["diabetes"], test_class, pos_label=0), 3))
print("train AUC:", round(roc_auc_score(train["diabetes"], train_prob), 3))
print("test AUC: ", round(roc_auc_score(test["diabetes"], test_prob), 3))

> R chose `tree_depth` 7 and `min_n` 23 (cross-validated AUC 0.775), a tree with
> 14 leaves: test accuracy 0.760, sensitivity 0.642, specificity 0.825; AUC 0.854
> on the training rows and 0.800 on the test rows. Python chose `max_depth` 5 and
> `min_samples_split` 30 (cross-validated AUC 0.767), 15 leaves: test accuracy
> 0.747, sensitivity 0.585, specificity 0.835; AUC 0.859 on the training rows and
> 0.782 on the test rows. The gap between training and test is far smaller than
> for the tree that nothing stopped, which was perfect on the training rows.

Cross-validation splits the training rows at random, and trees break ties
between equal splits in different ways, so R and Python do not choose the same
tree and their numbers differ.

**Question!** Compare the tuned tree with the basic tree of Part 2. Has the gap
between training and test performance closed? Why must the test rows play no
part in choosing `tree_depth` and `min_n`?

> **Learning points**
>
> - A grid search tries every combination of hyperparameter values and judges each by cross-validation on the training rows.
> - Hyperparameter tuning is part of training: the test set must never be used to choose hyperparameters, only to evaluate the final model, once.
> - Workflows (R) and pipelines (Python) keep the recipe and the model together, so the tuned model is fitted and evaluated exactly as it was tuned.

## Stretch

> **Stretch — tune the pruning instead**
>
> Leave `tree_depth` and `min_n` at their defaults and tune
> `cost_complexity` instead (in Python, `ccp_alpha`), over the values 0.0001,
> 0.001, 0.01 and 0.1. Which value is chosen, how many leaves has the tuned tree,
> and how does its test AUC compare with the tree tuned above?